Note to self: what is this model doing?

In Monte Carlo data I know the true label for each event. 

stopped\_muon = 1 means the muon stops inside the detector.  
stopped\_muon = 0 means the muon goes through the detector.  
stopped\_muon = -1 means the label is not defined and these events are ignored.

In real data this information does not exist.  
All I ever observe are DOM pulses: charge, time, and position.

The purpose of the stopped/through model is therefore $\emph{not}$ to determine the truth,
but to estimate how much an event $\emph{looks like}$ a stopped muon based on its pulse pattern.

Physically, a stopped muon produces a light pattern that ends inside the detector,
often with increased energy deposition near the end of the track.  
A through-going muon instead produces light more uniformly along its path and exits
the detector volume.

Using Monte Carlo events with a known stopping label, simple event-level features
are constructed from the observed DOM pulses, such as the total collected charge,
the number of pulses, and the temporal and spatial extent of the light pattern.

A logistic regression model is then trained to learn how combinations of these
pulse-derived features differ between stopped and through-going muons.

The model outputs a score
$$
P(\text{stopped} \mid \text{pulse features}),
$$
which quantifies how similar a given event’s pulse pattern is to that of a stopped
muon in the Monte Carlo sample.

In this code, the model is tested only on Monte Carlo events in order to examine
whether the chosen pulse-based features provide sufficient separation between
stopped and through-going muons in simulation.


In [ ]:
#[Cell 0] Load CSV files
import pandas as pd

# Read the CSV file
df_MC = pd.read_csv('/groups/icecube/holgerkc/Thesis_Analysis/old/MC_prediction_muon_noise_neutrino_02_02_2026.csv')
df_BS = pd.read_csv('/groups/icecube/holgerkc/Thesis_Analysis/old/Burnsample_prediction_muon_noise_neutrino_2022.csv')

Below the target label $y$ is defined as
$$
y =
\begin{cases}
1, & \text{if the muon is classified as a stopped muon}, \\
0, & \text{if the muon is classified as a through-going muon}.
\end{cases}
$$


In [ ]:
# [cell 1] Load labels + choose event list

import sqlite3
import pandas as pd

DB = "/groups/icecube/holgerkc/Thesis_Analysis/old/MC_pulsemap_muon_noise_neutrino_02_02_2026.db"

N_EVENTS = 10000

with sqlite3.connect(DB, timeout=10) as conn:
    df_truth = pd.read_sql_query("""
        SELECT event_no, stopped_muon
        FROM truth
        WHERE stopped_muon IN (0,1)
        LIMIT ?
    """, conn, params=(N_EVENTS,))

df_truth["y"] = df_truth["stopped_muon"].astype(int)
df_truth = df_truth[["event_no", "y"]]

print("Loaded labeled events:", len(df_truth))
print("Label counts:")
print(df_truth["y"].value_counts())
print("\nFirst 5 events:")
print(df_truth.head().to_string(index=False))



In [ ]:
# %% [cell 3] building minimal features. 

import sqlite3
import numpy as np
import pandas as pd
import time

DB = "/groups/icecube/holgerkc/Thesis_Analysis/old/MC_pulsemap_muon_noise_neutrino_02_02_2026.db"

rows = []
t0 = time.time()

with sqlite3.connect(DB, timeout=10) as conn:
    cur = conn.cursor()

    for i, (event_no, y) in enumerate(df_truth.itertuples(index=False), 1): #Here we loop over events, 1 at a time. i counts how many events we treated. 
        cur.execute("""
            SELECT charge, dom_time, dom_z 
            FROM SplitInIcePulses
            WHERE event_no = ?
        """, (int(event_no),)) # Fetches charge, time, and z-position of all pulses from SplitInIcePulses, for the current event_no. 

        pulses = cur.fetchall() # Just reads all events WITH pulses. If there are no pulses, we skip this event and move to the next one.
        if not pulses:
            continue

        arr = np.asarray(pulses, dtype=float) # Converts the list of pulses into a NumPy array for easier processing. Each row corresponds to a pulse,
                                              # and columns correspond to charge, time, and z-position as defined in the cur.execute() query.
                                              
        q = arr[:, 0]
        t = arr[:, 1]
        z = arr[:, 2]

        Q_tot = q.sum() # total charge
        N_hits = len(q) # number of pulses (hits)
        dt = t.max() - t.min() # duration of the event
        dz = z.max() - z.min() # vertical extent of the event

        z_cw = (q * z).sum() / Q_tot # charge-weighted mean z-position of the pulses, i.e. we expect that through-going muons will have a 
                                     # z_cw somewhere in the middle of the detector, while stopped muons will have a z_cw closer to the top or bottom.

        rows.append({
            "event_no": event_no,
            "y": y,
            "Q_tot": Q_tot,
            "N_hits": N_hits,
            "dt": dt,
            "dz": dz,
            "z_cw": z_cw,
        }) # We create a dictionary for each event with the computed features and the label y, and append it to the list of rows.

        if i % 1000 == 0:
            elapsed = time.time() - t0
            print(f"processed {i}/{len(df_truth)} events | {i/elapsed:.1f} events/s") # recording progress and processing speed every 1000 events
                                                                                  # so that we can monitor how long the feature extraction is taking. 

print("")
print("Example row: 1")
print(rows[1])

df = pd.DataFrame(rows) # Finally, we convert the list of dictionaries into a pandas DataFrame for easier analysis and modeling.
                        # Each row corresponds to an event, and columns correspond to the features and label.


print("\nFeature table:")
print(df.head().to_string(index=False)) #df.head() → the first 5 rows in the DataFrame (standard in pandas)


print("\nLabel counts in df:")
print(df["y"].value_counts())


 We write StandardScaler manually, just to show how it works. In practice, you would use the one from sklearn.preprocessing. This is what is called a Transformer. A Transformer is a class that has a fit() method to compute parameters from the training data, and a transform() method to apply the transformation to any data (training or test). In this case, our StandardScaler_manuel computes the mean and standard deviation of each feature from the training data in fit(), and then uses those parameters to standardize the features in transform().

In [ ]:

class StandardScaler_manuel:
    def fit(self, X, y=None):
        self.mu = X.mean(axis=0)
        self.sigma = X.std(axis=0)
        return self

    def transform(self, X):
        return (X - self.mu) / self.sigma 



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler #currently replaced by our own identical implementation, just for demonstration purposes.
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import numpy as np

X = df[["Q_tot", "N_hits", "dt", "dz", "z_cw"]].values
y = df["y"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)


model = Pipeline([
    ("scaler", StandardScaler_manuel()),
    ("clf", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

p = model.predict_proba(X_test)[:, 1]
yhat = (p >= 0.5).astype(int)

print("AUC:", roc_auc_score(y_test, p))
print("Confusion matrix:\n", confusion_matrix(y_test, yhat))
print(classification_report(y_test, yhat, digits=4))

coef = model.named_steps["clf"].coef_[0]
names = ["Q_tot", "N_hits", "dt", "dz", "z_cw"]
print("\nFeature weights (logistic regression):")
for n, c in sorted(zip(names, coef), key=lambda t: abs(t[1]), reverse=True):
    print(f"{n:6s}: {c:+.4f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# scores for all events in df (MC)
X_all = df[["Q_tot", "N_hits", "dt", "dz", "z_cw"]].values
p_stop = model.predict_proba(X_all)[:, 1]

p0 = p_stop[df["y"].values == 0]  # through
p1 = p_stop[df["y"].values == 1]  # stopped

bins = np.linspace(0, 1, 101)

plt.figure()
plt.hist(p0, bins=bins, histtype="step", density=True, log=False, label="Through Muon")
plt.hist(p1, bins=bins, histtype="step", density=True, log=False, label="Stopped Muon")
plt.xlabel("Probability")
plt.ylabel("Density")
plt.title(f"Classify Stopped and Through Muon [{len(df):,} events used]")
plt.legend()
plt.tight_layout()
plt.show()
